<a href="https://colab.research.google.com/github/paulinepiccio/judging-by-the-cover/blob/main/notebooks/prizes02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import auth

auth.authenticate_user()

PROJECT_ID = "judging-by-the-cover"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

In [ ]:
GITHUB_RAW_URL = "https://raw.githubusercontent.com/paulinepiccio/judging-by-the-cover/main/data/raw"

files = {
    'goncourt': f"{GITHUB_RAW_URL}/goncourt_raw.csv",
    'renaudot': f"{GITHUB_RAW_URL}/renaudot_raw.csv",
    'booker': f"{GITHUB_RAW_URL}/booker_raw.csv",
    'pulitzer': f"{GITHUB_RAW_URL}/pulitzer_raw.csv",
    'nobel': f"{GITHUB_RAW_URL}/nobel_raw.csv",
    'akutagawa': f"{GITHUB_RAW_URL}/akutagawa_raw.csv",
    'naoki': f"{GITHUB_RAW_URL}/naoki_raw.csv",
}

dfs = {name: pd.read_csv(url) for name, url in files.items()}

for name, df in dfs.items():
    print(f"{name}: {df.shape[0]} rows, columns: {list(df.columns)}")

goncourt: 123 rows, columns: ['prize', 'country', 'year', 'author', 'book_title', 'publisher', 'publisher_cumulative_count', 'notes']
renaudot: 100 rows, columns: ['prize', 'country', 'year', 'author', 'book_title', 'publisher', 'publisher_cumulative_count', 'notes']
booker: 60 rows, columns: ['prize', 'country', 'year', 'author', 'author_country', 'book_title']
pulitzer: 79 rows, columns: ['prize', 'country', 'year', 'author', 'author_country', 'book_title', 'publisher', 'notes']
nobel: 131 rows, columns: ['prize', 'country', 'year', 'author', 'author_country', 'book_title', 'publisher', 'notes']
akutagawa: 222 rows, columns: ['prize', 'country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']
naoki: 233 rows, columns: ['prize', 'country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']


In [ ]:
def harmonize_schema(df, name):
    df = df.copy()

    if 'country' in df.columns:
        df = df.rename(columns={'country': 'prize_country'})

    if 'session' not in df.columns:
        df['session'] = None

    if 'author_country' not in df.columns:
        df['author_country'] = None

    if 'publisher' not in df.columns:
        df['publisher'] = None

    if 'book_title' not in df.columns:
        df['book_title'] = None

    if 'notes' not in df.columns:
        df['notes'] = None

    cols_to_drop = [c for c in df.columns if c not in [
        'prize', 'prize_country', 'year', 'session', 'author',
        'author_country', 'book_title', 'publisher', 'notes'
    ]]
    if cols_to_drop:
        print(f"  Dropping extra columns from {name}: {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)

    target_order = ['prize', 'prize_country', 'year', 'session', 'author',
                    'author_country', 'book_title', 'publisher', 'notes']
    df = df[target_order]

    return df

dfs_clean = {name: harmonize_schema(df, name) for name, df in dfs.items()}

for name, df in dfs_clean.items():
    print(f"{name}: {df.shape[0]} rows, columns: {list(df.columns)}")

  Dropping extra columns from goncourt: ['publisher_cumulative_count']
  Dropping extra columns from renaudot: ['publisher_cumulative_count']
goncourt: 123 rows, columns: ['prize', 'prize_country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']
renaudot: 100 rows, columns: ['prize', 'prize_country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']
booker: 60 rows, columns: ['prize', 'prize_country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']
pulitzer: 79 rows, columns: ['prize', 'prize_country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']
nobel: 131 rows, columns: ['prize', 'prize_country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']
akutagawa: 222 rows, columns: ['prize', 'prize_country', 'year', 'session', 'author', 'author_country', 'book_title', 'publisher', 'notes']
naoki: 233 rows, columns: ['pr

In [ ]:
schemas = {name: list(df.columns) for name, df in dfs_clean.items()}
for name, df in dfs_clean.items():
    print(f"{name}: {df.shape}")

goncourt: (123, 9)
renaudot: (100, 9)
booker: (60, 9)
pulitzer: (79, 9)
nobel: (131, 9)
akutagawa: (222, 9)
naoki: (233, 9)


In [ ]:
COUNTRY_MAPPING = {
    'France': 'France',
    'Royaume-Uni': 'UK',
    'United Kingdom': 'UK',
    'UK': 'UK',
    'États-Unis': 'USA',
    'United States': 'USA',
    'USA': 'USA',
    'Japon': 'Japan',
    'Japan': 'Japan',
    'Allemagne': 'Germany',
    'Empire allemand': 'Germany',
    'Germany': 'Germany',
    'République fédérale allemande': 'Germany',
    'Italie': 'Italy',
    'Espagne': 'Spain',
    "Royaume d'Espagne": 'Spain',
    'Royaume du Danemark': 'Denmark',
    'Danemark': 'Denmark',
    'Norvège': 'Norway',
    'Suède': 'Sweden',
    'Belgique': 'Belgium',
    'Pologne': 'Poland',
    'Royaume de Pologne': 'Poland',
    'Russie': 'Russia',
    'Empire russe': 'Russia',
    'Union soviétique': 'USSR',
    'URSS': 'USSR',
    'Inde': 'India',
    'Raj britannique': 'India',
    'Chine': 'China',
    'Corée du Sud': 'South Korea',
    'Israël': 'Israel',
    'Grèce': 'Greece',
    'Autriche': 'Austria',
    'Hongrie': 'Hungary',
    'Roumanie': 'Romania',
    'Tchéquie': 'Czech Republic',
    'Yougoslavie': 'Yugoslavia',
    'Irlande': 'Ireland',
    'Canada': 'Canada',
    'Australie': 'Australia',
    'Nouvelle-Zélande': 'New Zealand',
    'Afrique du Sud': 'South Africa',
    'Égypte': 'Egypt',
    'Turquie': 'Turkey',
    'Mexique': 'Mexico',
    'Chili': 'Chile',
    'Colombie': 'Colombia',
    'Argentine': 'Argentina',
    'Pérou': 'Peru',
    'Guatemala': 'Guatemala',
    'Nigéria': 'Nigeria',
    'Kenya': 'Kenya',
    'Tanzanie': 'Tanzania',
    'Saint-Christophe-et-Niévès': 'Saint Kitts and Nevis',
    'Trinité-et-Tobago': 'Trinidad and Tobago',
    'Biélorussie': 'Belarus',
}

def normalize_country(country):
    if pd.isna(country) or country is None or country == '':
        return None
    country = str(country).strip()
    return COUNTRY_MAPPING.get(country, country)

In [ ]:
print(dfs_clean['booker']['author_country'].value_counts().head(20))

author_country
ENG         24
AUS          5
IRL          4
CAN          3
IND          3
RSA          2
USA          2
ENG  IRL     2
ZAF          2
NZL          2
UK  TTO      1
WAL          1
NGR          1
SCO          1
CAN  SRI     1
UK  DE       1
JAM          1
UK           1
SCO  USA     1
SRI          1
Name: count, dtype: int64


In [ ]:
BOOKER_CODE_MAPPING = {
    'ENG': 'UK',
    'SCO': 'UK',
    'WAL': 'UK',
    'UK': 'UK',
    'IRL': 'Ireland',
    'AUS': 'Australia',
    'CAN': 'Canada',
    'IND': 'India',
    'USA': 'USA',
    'RSA': 'South Africa',
    'ZAF': 'South Africa',
    'NZL': 'New Zealand',
    'NGR': 'Nigeria',
    'JAM': 'Jamaica',
    'SRI': 'Sri Lanka',
    'TTO': 'Trinidad and Tobago',
    'DE': 'Germany',
}

def parse_booker_country(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    parts = s.split()
    if not parts:
        return None
    first_code = parts[0]
    return BOOKER_CODE_MAPPING.get(first_code, first_code)

dfs_clean['booker']['author_country'] = dfs_clean['booker']['author_country'].apply(parse_booker_country)

print("Booker author_country after normalization:")
print(dfs_clean['booker']['author_country'].value_counts())

Booker author_country after normalization:
author_country
UK              32
Australia        5
Canada           5
South Africa     4
Ireland          4
India            3
New Zealand      2
USA              2
Nigeria          1
Jamaica          1
Sri Lanka        1
Name: count, dtype: int64


In [ ]:
for name, df in dfs_clean.items():
    if name != 'booker':
        df['author_country'] = df['author_country'].apply(normalize_country)
    df['prize_country'] = df['prize_country'].apply(normalize_country)

print("All unique author_country values after normalization:")
all_countries = pd.concat([df['author_country'] for df in dfs_clean.values()])
print(sorted(all_countries.dropna().unique()))

All unique author_country values after normalization:
["Allemagne de l'Ouest", 'Allemand naturalisé', 'Australia', 'Austria', 'Belarus', 'Belgium', 'Bulgarie', 'Canada', 'Chile', 'China', 'Colombia', 'Denmark', 'Egypt', 'Finlande', 'France', 'Germany', 'Greece', 'Guatemala', 'Hungary', 'India', 'Ireland', 'Islande', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Maurice', 'Mexico', 'New Zealand', 'Nigeria', 'Norway', 'Peru', 'Poland', 'Portugal', 'Romania', 'Russe blanc', 'Sainte-Lucie', 'South Africa', 'South Korea', 'Spain', 'Sri Lanka', 'Suisse', 'Sweden', 'Tanzania', 'Tchécoslovaquie', 'Turkey', 'UK', 'USA', 'USSR', 'Yugoslavia']


In [ ]:
all_dfs_concat = pd.concat([df.assign(source=name) for name, df in dfs_clean.items()], ignore_index=True)

print(all_dfs_concat[all_dfs_concat['author_country'] == 'Allemand naturalisé'])

print(all_dfs_concat[all_dfs_concat['author_country'] == 'Russe blanc'])

                prize prize_country  year session         author  \
406  Nobel Literature        Sweden  1946    None  Hermann Hesse   

          author_country book_title publisher notes source  
406  Allemand naturalisé        NaN       NaN   NaN  nobel  
                prize prize_country  year session        author  \
396  Nobel Literature        Sweden  1933    None  Ivan Bounine   

    author_country book_title publisher notes source  
396    Russe blanc        NaN       NaN   NaN  nobel  


In [ ]:
ADDITIONAL_MAPPING = {
    "Allemagne de l'Ouest": 'West Germany',
    'Bulgarie': 'Bulgaria',
    'Finlande': 'Finland',
    'Islande': 'Iceland',
    'Maurice': 'Mauritius',
    'Sainte-Lucie': 'Saint Lucia',
    'Suisse': 'Switzerland',
    'Tchécoslovaquie': 'Czechoslovakia',
    'Russe blanc': 'Russia',
    'Allemand naturalisé': 'Germany',
}

COUNTRY_MAPPING.update(ADDITIONAL_MAPPING)

for name, df in dfs_clean.items():
    df['author_country'] = df['author_country'].apply(normalize_country)

all_countries = pd.concat([df['author_country'] for df in dfs_clean.values()])
print(sorted(all_countries.dropna().unique()))

['Australia', 'Austria', 'Belarus', 'Belgium', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Czechoslovakia', 'Denmark', 'Egypt', 'Finland', 'France', 'Germany', 'Greece', 'Guatemala', 'Hungary', 'Iceland', 'India', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Mauritius', 'Mexico', 'New Zealand', 'Nigeria', 'Norway', 'Peru', 'Poland', 'Portugal', 'Romania', 'Russia', 'Saint Lucia', 'South Africa', 'South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Switzerland', 'Tanzania', 'Turkey', 'UK', 'USA', 'USSR', 'West Germany', 'Yugoslavia']


In [ ]:
ADDITIONAL_MAPPING_V2 = {
    "Allemagne de l'Ouest": 'Germany',
    'West Germany': 'Germany',
    'Bulgarie': 'Bulgaria',
    'Finlande': 'Finland',
    'Islande': 'Iceland',
    'Maurice': 'Mauritius',
    'Sainte-Lucie': 'Saint Lucia',
    'Suisse': 'Switzerland',
    'Tchécoslovaquie': 'Czech Republic',
    'Russe blanc': 'Russia',
    'Allemand naturalisé': 'Germany',
    'USSR': 'Russia',
    'Union soviétique': 'Russia',
    'URSS': 'Russia',
    'Yugoslavia': 'Serbia',
    'Yougoslavie': 'Serbia',
}

COUNTRY_MAPPING.update(ADDITIONAL_MAPPING_V2)

for name, df in dfs_clean.items():
    df['author_country'] = df['author_country'].apply(normalize_country)

all_countries = pd.concat([df['author_country'] for df in dfs_clean.values()])
final_list = sorted(all_countries.dropna().unique())
print(f"Total unique countries: {len(final_list)}")
print(final_list)

Total unique countries: 47
['Australia', 'Austria', 'Belarus', 'Belgium', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Czechoslovakia', 'Denmark', 'Egypt', 'Finland', 'France', 'Germany', 'Greece', 'Guatemala', 'Hungary', 'Iceland', 'India', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Mauritius', 'Mexico', 'New Zealand', 'Nigeria', 'Norway', 'Peru', 'Poland', 'Portugal', 'Romania', 'Russia', 'Saint Lucia', 'Serbia', 'South Africa', 'South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Switzerland', 'Tanzania', 'Turkey', 'UK', 'USA']


In [ ]:
all_dfs_concat = pd.concat([df.assign(source=name) for name, df in dfs_clean.items()], ignore_index=True)
print(all_dfs_concat[all_dfs_concat['author_country'] == 'Czechoslovakia'][['source', 'year', 'author', 'author_country']])

    source  year            author  author_country
448  nobel  1984  Jaroslav Seifert  Czechoslovakia


In [ ]:
COUNTRY_MAPPING['Czechoslovakia'] = 'Czech Republic'

for name, df in dfs_clean.items():
    df['author_country'] = df['author_country'].apply(normalize_country)
print(sorted(pd.concat([df['author_country'] for df in dfs_clean.values()]).dropna().unique()))

['Australia', 'Austria', 'Belarus', 'Belgium', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Czech Republic', 'Denmark', 'Egypt', 'Finland', 'France', 'Germany', 'Greece', 'Guatemala', 'Hungary', 'Iceland', 'India', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Mauritius', 'Mexico', 'New Zealand', 'Nigeria', 'Norway', 'Peru', 'Poland', 'Portugal', 'Romania', 'Russia', 'Saint Lucia', 'Serbia', 'South Africa', 'South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Switzerland', 'Tanzania', 'Turkey', 'UK', 'USA']


In [ ]:
print("Goncourt - author_country nulls:", dfs_clean['goncourt']['author_country'].isna().sum())
print("Renaudot - author_country nulls:", dfs_clean['renaudot']['author_country'].isna().sum())

Goncourt - author_country nulls: 123
Renaudot - author_country nulls: 100


In [ ]:
dfs_clean['goncourt']['author_country'] = 'France'
dfs_clean['renaudot']['author_country'] = 'France'

NATIONALITY_PATTERNS = {
    'marocain': 'Morocco',
    'marocaine': 'Morocco',
    'sénégalais': 'Senegal',
    'sénégalaise': 'Senegal',
    'algérien': 'Algeria',
    'algérienne': 'Algeria',
    'belge': 'Belgium',
    'libanais': 'Lebanon',
    'libanaise': 'Lebanon',
    'russe': 'Russia',
    'afghan': 'Afghanistan',
    'haïtien': 'Haiti',
    'haïtienne': 'Haiti',
    'congolais': 'Congo',
    'congolaise': 'Congo',
    'guinéen': 'Guinea',
    'guinéenne': 'Guinea',
    'tunisien': 'Tunisia',
    'tunisienne': 'Tunisia',
    'malien': 'Mali',
    'malienne': 'Mali',
    'américain': 'USA',
    'américaine': 'USA',
    'canadien': 'Canada',
    'canadienne': 'Canada',
    'suisse': 'Switzerland',
    'italien': 'Italy',
    'italienne': 'Italy',
}

def detect_nationality_from_notes(notes):
    if pd.isna(notes):
        return None
    notes_lower = str(notes).lower()
    for pattern, country in NATIONALITY_PATTERNS.items():
        if pattern in notes_lower:
            return country
    return None

for prize_name in ['goncourt', 'renaudot']:
    df = dfs_clean[prize_name]
    df['detected_nationality'] = df['notes'].apply(detect_nationality_from_notes)

    mask = df['detected_nationality'].notna()
    df.loc[mask, 'author_country'] = df.loc[mask, 'detected_nationality']

    print(f"\n{prize_name.upper()}: {mask.sum()} laureates identified as non-French")
    print(df[mask][['year', 'author', 'notes', 'author_country']])

    df.drop(columns=['detected_nationality'], inplace=True)


GONCOURT: 11 laureates identified as non-French
     year                author  \
0    1903      John-Antoine Nau   
34   1937      Charles Plisnier   
55   1958        Francis Walder   
70   1973       Jacques Chessex   
76   1979      Antonine Maillet   
84   1987     Tahar Ben Jelloun   
90   1993          Amin Maalouf   
102  2005    François Weyergans   
113  2016         Leïla Slimani   
118  2021  Mohamed Mbougar Sarr   
121  2024           Kamel Daoud   

                                                 notes author_country  
0    Franco-américain d'expression française, premi...            USA  
34              Auteur belge, premier lauréat étranger        Belgium  
55                                        Auteur belge        Belgium  
70                              Premier lauréat suisse    Switzerland  
76   Premier lauréat canadien et premier lauréat no...         Canada  
84                Premier lauréat marocain et africain        Morocco  
90                        

In [ ]:
for author in ['Patrick Chamoiseau', 'Andreï Makine', 'Marie NDiaye', 'Romain Gary']:
    print(dfs_clean['goncourt'][dfs_clean['goncourt']['author'] == author][['year', 'author', 'notes', 'author_country']])
    print()

    year              author notes author_country
89  1992  Patrick Chamoiseau   NaN         France

    year         author                   notes author_country
92  1995  Andreï Makine  Également prix Médicis         France

     year        author notes author_country
106  2009  Marie NDiaye   NaN         France

    year       author notes author_country
53  1956  Romain Gary   NaN         France



In [ ]:
df_unified = pd.concat(dfs_clean.values(), ignore_index=True)

print(f"Total rows: {len(df_unified)}")
print(f"Rows per prize:")
print(df_unified['prize'].value_counts())
print(df_unified.head())
print(df_unified.tail())

Total rows: 948
Rows per prize:
prize
Naoki               233
Akutagawa           222
Nobel Literature    131
Goncourt            123
Renaudot            100
Pulitzer Fiction     79
Booker               60
Name: count, dtype: int64
      prize prize_country  year session                  author  \
0  Goncourt        France  1903    None        John-Antoine Nau   
1  Goncourt        France  1904    None             Léon Frapié   
2  Goncourt        France  1905    None          Claude Farrère   
3  Goncourt        France  1906    None  Jérôme et Jean Tharaud   
4  Goncourt        France  1907    None           Émile Moselly   

  author_country                                         book_title  \
0            USA                                      Force ennemie   
1         France                                      La Maternelle   
2         France                                      Les Civilisés   
3         France                       Dingley, l'illustre écrivain   
4         

In [ ]:
print(f"Total rows: {len(df_unified)}")
print(f"Rows with laureate (author not null): {df_unified['author'].notna().sum()}")
print(f"Rows without laureate: {df_unified['author'].isna().sum()}")


print(df_unified.isnull().sum())

print(f"Min year: {df_unified['year'].min()}")
print(f"Max year: {df_unified['year'].max()}")

print(f"Prize countries: {df_unified['prize_country'].nunique()}")
print(f"Author countries: {df_unified['author_country'].nunique()}")

dupes = df_unified[df_unified.duplicated(subset=['prize', 'year', 'session', 'author'], keep=False)]
print(f"Potential duplicates: {len(dupes)}")
if len(dupes) > 0:
    print(dupes.sort_values(['prize', 'year']))

Total rows: 948
Rows with laureate (author not null): 875
Rows without laureate: 73
prize               0
prize_country       0
year                0
session           493
author             73
author_country      4
book_title        200
publisher         536
notes             696
dtype: int64
Min year: 1901
Max year: 2025
Prize countries: 5
Author countries: 54
Potential duplicates: 2
         prize prize_country  year session           author author_country  \
528  Akutagawa         Japan  1953      H1  Shōtarō Yasuoka          Japan   
529  Akutagawa         Japan  1953      H1  Shōtarō Yasuoka          Japan   

                           book_title publisher notes  
528  Bad Company (悪い仲間, Warui Nakama)     Gunzō   NaN  
529        Inki na Tanoshimi (陰気な愉しみ)   Shinchō   NaN  


In [ ]:
mask_keep = ~((df_unified['prize'] == 'Akutagawa') &
              (df_unified['year'] == 1953) &
              (df_unified['session'] == 'H1') &
              (df_unified['book_title'].str.contains('Inki na Tanoshimi', na=False)))

df_unified.loc[(df_unified['prize'] == 'Akutagawa') &
               (df_unified['year'] == 1953) &
               (df_unified['session'] == 'H1') &
               (df_unified['book_title'].str.contains('Bad Company', na=False)),
               'book_title'] = 'Bad Company (悪い仲間, Warui Nakama); Inki na Tanoshimi (陰気な愉しみ)'

df_unified = df_unified[mask_keep].reset_index(drop=True)

In [ ]:
print(df_unified[df_unified['author_country'].isna()])

                prize prize_country  year session author author_country  \
376  Nobel Literature        Sweden  1914    None    NaN           None   
381  Nobel Literature        Sweden  1918    None    NaN           None   
398  Nobel Literature        Sweden  1935    None    NaN           None   
403  Nobel Literature        Sweden  1940    None    NaN           None   

    book_title publisher             notes  
376        NaN       NaN  No prize awarded  
381        NaN       NaN  No prize awarded  
398        NaN       NaN  No prize awarded  
403        NaN       NaN  No prize awarded  


In [ ]:
print(df_unified.groupby('prize')['session'].apply(lambda x: x.notna().sum()))

prize
Akutagawa           220
Booker                0
Goncourt              0
Naoki               233
Nobel Literature      0
Pulitzer Fiction      0
Renaudot              0
Name: session, dtype: int64


In [ ]:
df_unified.to_csv('all_prizes_processed.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_unified)} rows")

Saved 946 rows


In [33]:
prizes_data = [
    {'prize_id': 'goncourt', 'name': 'Prix Goncourt', 'organizing_country': 'France', 'founding_year': 1903,
     'description': 'Most prestigious French literary prize, awarded annually for a novel.'},
    {'prize_id': 'renaudot', 'name': 'Prix Renaudot', 'organizing_country': 'France', 'founding_year': 1926,
     'description': 'French literary prize created as an alternative to the Goncourt.'},
    {'prize_id': 'booker', 'name': 'Booker Prize', 'organizing_country': 'UK', 'founding_year': 1969,
     'description': 'Major literary prize for fiction in English.'},
    {'prize_id': 'pulitzer', 'name': 'Pulitzer Prize for Fiction', 'organizing_country': 'USA', 'founding_year': 1948,
     'description': 'American literary prize for fiction by American authors.'},
    {'prize_id': 'nobel', 'name': 'Nobel Prize in Literature', 'organizing_country': 'Sweden', 'founding_year': 1901,
     'description': 'International prize for outstanding contribution to literature.'},
    {'prize_id': 'akutagawa', 'name': 'Akutagawa Prize', 'organizing_country': 'Japan', 'founding_year': 1935,
     'description': 'Japanese literary prize for promising new authors, awarded biannually.'},
    {'prize_id': 'naoki', 'name': 'Naoki Prize', 'organizing_country': 'Japan', 'founding_year': 1935,
     'description': 'Japanese literary prize for popular fiction, awarded biannually.'},
]

df_prizes = pd.DataFrame(prizes_data)
print(df_prizes)

    prize_id                        name organizing_country  founding_year  \
0   goncourt               Prix Goncourt             France           1903   
1   renaudot               Prix Renaudot             France           1926   
2     booker                Booker Prize                 UK           1969   
3   pulitzer  Pulitzer Prize for Fiction                USA           1948   
4      nobel   Nobel Prize in Literature             Sweden           1901   
5  akutagawa             Akutagawa Prize              Japan           1935   
6      naoki                 Naoki Prize              Japan           1935   

                                         description  
0  Most prestigious French literary prize, awarde...  
1  French literary prize created as an alternativ...  
2       Major literary prize for fiction in English.  
3  American literary prize for fiction by America...  
4  International prize for outstanding contributi...  
5  Japanese literary prize for promising new 

In [34]:
all_countries = pd.concat([
    df_unified['prize_country'],
    df_unified['author_country']
]).dropna().unique()

CONTINENT_MAPPING = {
    'France': 'Europe', 'UK': 'Europe', 'Germany': 'Europe', 'Italy': 'Europe',
    'Spain': 'Europe', 'Portugal': 'Europe', 'Belgium': 'Europe', 'Netherlands': 'Europe',
    'Sweden': 'Europe', 'Norway': 'Europe', 'Denmark': 'Europe', 'Finland': 'Europe',
    'Iceland': 'Europe', 'Ireland': 'Europe', 'Switzerland': 'Europe', 'Austria': 'Europe',
    'Poland': 'Europe', 'Czech Republic': 'Europe', 'Hungary': 'Europe', 'Romania': 'Europe',
    'Bulgaria': 'Europe', 'Greece': 'Europe', 'Serbia': 'Europe', 'Belarus': 'Europe',
    'Russia': 'Europe', 'Turkey': 'Europe',
    'USA': 'North America', 'Canada': 'North America', 'Mexico': 'North America',
    'Jamaica': 'North America', 'Saint Lucia': 'North America',
    'Chile': 'South America', 'Colombia': 'South America', 'Peru': 'South America',
    'Guatemala': 'South America',
    'Japan': 'Asia', 'China': 'Asia', 'India': 'Asia', 'South Korea': 'Asia',
    'Sri Lanka': 'Asia', 'Israel': 'Asia', 'Afghanistan': 'Asia', 'Lebanon': 'Asia',
    'Egypt': 'Africa', 'Nigeria': 'Africa', 'Kenya': 'Africa', 'Tanzania': 'Africa',
    'South Africa': 'Africa', 'Morocco': 'Africa', 'Algeria': 'Africa',
    'Senegal': 'Africa', 'Congo': 'Africa', 'Guinea': 'Africa', 'Mali': 'Africa',
    'Tunisia': 'Africa', 'Mauritius': 'Africa',
    'Australia': 'Oceania', 'New Zealand': 'Oceania',
    'Trinidad and Tobago': 'North America',
    'Haiti': 'North America',
}

countries_data = []
for i, country in enumerate(sorted(all_countries), 1):
    countries_data.append({
        'country_id': f'C{i:03d}',
        'name': country,
        'continent': CONTINENT_MAPPING.get(country, 'Unknown')
    })

df_countries = pd.DataFrame(countries_data)
print(df_countries)
print(f"Unknown continents: {df_countries[df_countries['continent'] == 'Unknown']}")

   country_id            name      continent
0        C001         Algeria         Africa
1        C002       Australia        Oceania
2        C003         Austria         Europe
3        C004         Belarus         Europe
4        C005         Belgium         Europe
5        C006        Bulgaria         Europe
6        C007          Canada  North America
7        C008           Chile  South America
8        C009           China           Asia
9        C010        Colombia  South America
10       C011           Congo         Africa
11       C012  Czech Republic         Europe
12       C013         Denmark         Europe
13       C014           Egypt         Africa
14       C015         Finland         Europe
15       C016          France         Europe
16       C017         Germany         Europe
17       C018          Greece         Europe
18       C019       Guatemala  South America
19       C020          Guinea         Africa
20       C021           Haiti  North America
21       C

In [36]:
df_authors_raw = df_unified[df_unified['author'].notna()][['author', 'author_country']].copy()
df_authors_unique = df_authors_raw.drop_duplicates(subset=['author']).reset_index(drop=True)

df_authors_unique = df_authors_unique.merge(
    df_countries[['country_id', 'name']].rename(columns={'name': 'author_country'}),
    on='author_country',
    how='left'
)

df_authors_unique['author_id'] = ['A' + str(i).zfill(4) for i in range(1, len(df_authors_unique) + 1)]
df_authors = df_authors_unique[['author_id', 'author', 'country_id']].rename(columns={'author': 'name'})

print(f"Total unique authors: {len(df_authors)}")
print(df_authors.head())

Total unique authors: 847
  author_id                    name country_id
0     A0001        John-Antoine Nau       C054
1     A0002             Léon Frapié       C016
2     A0003          Claude Farrère       C016
3     A0004  Jérôme et Jean Tharaud       C016
4     A0005           Émile Moselly       C016


In [40]:
df_laureates = df_unified.copy()

df_laureates = df_laureates.merge(
    df_authors[['author_id', 'name']].rename(columns={'name': 'author'}),
    on='author',
    how='left'
)

prize_id_mapping = {
    'Goncourt': 'goncourt',
    'Renaudot': 'renaudot',
    'Booker': 'booker',
    'Pulitzer Fiction': 'pulitzer',
    'Nobel Literature': 'nobel',
    'Akutagawa': 'akutagawa',
    'Naoki': 'naoki',
}
df_laureates['prize_id'] = df_laureates['prize'].map(prize_id_mapping)

df_laureates['laureate_id'] = ['L' + str(i).zfill(4) for i in range(1, len(df_laureates) + 1)]

df_laureates_final = df_laureates[[
    'laureate_id', 'prize_id', 'author_id', 'year', 'session',
    'book_title', 'publisher', 'notes'
]]

print(f"Total laureates: {len(df_laureates_final)}")
print(df_laureates_final.head())

Total laureates: 946
  laureate_id  prize_id author_id  year session  \
0       L0001  goncourt     A0001  1903    None   
1       L0002  goncourt     A0002  1904    None   
2       L0003  goncourt     A0003  1905    None   
3       L0004  goncourt     A0004  1906    None   
4       L0005  goncourt     A0005  1907    None   

                                          book_title         publisher  \
0                                      Force ennemie          La Plume   
1                                      La Maternelle      Albin Michel   
2                                      Les Civilisés   Paul Ollendorff   
3                       Dingley, l'illustre écrivain  Édouard Pelletan   
4  Terres lorraines et  Jean des Brebis ou le Liv...              Plon   

                                               notes  
0  Franco-américain d'expression française, premi...  
1                                                NaN  
2                                                NaN  
3      

In [41]:
print(f"prizes: {len(df_prizes)} rows")
print(f"countries: {len(df_countries)} rows")
print(f"authors: {len(df_authors)} rows")
print(f"laureates: {len(df_laureates_final)} rows")

orphan_authors = df_laureates_final[
    df_laureates_final['author_id'].notna() &
    ~df_laureates_final['author_id'].isin(df_authors['author_id'])
]
print(f"Orphan author_ids in laureates: {len(orphan_authors)}")

orphan_prizes = df_laureates_final[~df_laureates_final['prize_id'].isin(df_prizes['prize_id'])]
print(f"Orphan prize_ids in laureates: {len(orphan_prizes)}")

orphan_countries = df_authors[df_authors['country_id'].notna() &
                              ~df_authors['country_id'].isin(df_countries['country_id'])]
print(f"Orphan country_ids in authors: {len(orphan_countries)}")

prizes: 7 rows
countries: 54 rows
authors: 847 rows
laureates: 946 rows
Orphan author_ids in laureates: 0
Orphan prize_ids in laureates: 0
Orphan country_ids in authors: 0


In [44]:
DATASET_ID = "literary_prizes"

def upload_to_bigquery(df, table_name, dataset_id=DATASET_ID):
    table_ref = client.dataset(dataset_id).table(table_name)

    job_config = bigquery.LoadJobConfig(
        write_disposition='WRITE_TRUNCATE',
        autodetect=True,
    )

    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()

    table = client.get_table(table_ref)

upload_to_bigquery(df_prizes, 'prizes')
upload_to_bigquery(df_countries, 'countries')
upload_to_bigquery(df_authors, 'authors')
upload_to_bigquery(df_laureates_final, 'laureates')